In [1]:
from pathlib import Path
import warnings
import copy
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torchvision import models, transforms

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__)

device: cpu | torch 2.14.1+cpu


In [2]:
BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "cv" / "processed"
LABELS_CSV = BASE_DIR / "data" / "cv" / "raw" / "solution.csv"
OUTPUTS_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
CACHE_DIR = OUTPUTS_DIR / "feature_cache"
FRAMES30_DIR = (
    PROCESSED_DIR / "frames_effnet30"
)  # New directory; previous frames remain untouched
for p in (FIGURES_DIR, CACHE_DIR, FRAMES30_DIR):
    p.mkdir(parents=True, exist_ok=True)

In [3]:
ldf = pd.read_csv(LABELS_CSV, dtype={"id": str})
lm = dict(zip(ldf["id"].str.zfill(5), ldf["target"]))

total, mismatch = 0, []
for s in ["train", "val", "test"]:
    d = next(
        p
        for p in [PROCESSED_DIR / f"{s}_v2", PROCESSED_DIR / f"{s}-v2"]
        if (p / "positive").exists()
    )
    for cls, folder_label in [("negative", 0), ("positive", 1)]:
        for f in (d / cls).glob("*.mp4"):
            total += 1
            vid = f.stem.zfill(5)
            if float(lm[vid]) != folder_label:
                mismatch.append((s, vid))

print("total videos:", total, "| mismatches:", len(mismatch))
print(mismatch)

total videos: 150 | mismatches: 0
[]


In [4]:
N_FRAMES = 30

def find_split(name):
    for cand in (f"{name}_v2", f"{name}-v2"):
        d = PROCESSED_DIR / cand
        if (d / "positive").exists():
            return d
    raise FileNotFoundError(
        f"Neither {name}_v2 nor {name}-v2 was found in {PROCESSED_DIR}"
    )


SPLIT_DIRS = {s: find_split(s) for s in ("train", "val", "test")}
print(SPLIT_DIRS)

ldf = pd.read_csv(LABELS_CSV, dtype={"id": str})
group_map = dict(zip(ldf["id"].str.zfill(5), ldf["group"]))

rows = []
for split, d in SPLIT_DIRS.items():
    for cls, label in [("negative", 0), ("positive", 1)]:
        for f in sorted((d / cls).glob("*.mp4")):
            vid = f.stem.zfill(5)
            rows.append(
                dict(
                    id=vid,
                    split=split,
                    label=label,
                    group=group_map.get(vid, -1),
                    path=f,
                )
            )
meta = pd.DataFrame(rows)

assert meta["id"].is_unique, "A video appears in more than one split!"
assert (meta["group"] >= 0).all(), "Some videos are missing from solution.csv"
print(meta.groupby(["split", "label"]).size().unstack())


{'train': WindowsPath('C:/Users/98936/Desktop/incident-sentiment-analyzer/data/cv/processed/train_v2'), 'val': WindowsPath('C:/Users/98936/Desktop/incident-sentiment-analyzer/data/cv/processed/val_v2'), 'test': WindowsPath('C:/Users/98936/Desktop/incident-sentiment-analyzer/data/cv/processed/test_v2')}
label   0   1
split        
test    7  16
train  35  70
val     8  14


In [5]:
def extract_frames(video_path, out_dir, n=N_FRAMES):
    # Skip extraction if the output directory already contains the exact number of frames
    if out_dir.exists() and len(list(out_dir.glob("*.jpg"))) == n:
        return True
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return False
    ok_count = 0
    for i, idx in enumerate(np.linspace(0, total - 1, n).astype(int)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            cv2.imwrite(str(out_dir / f"frame_{i:03d}.jpg"), frame)
            ok_count += 1
    cap.release()
    return ok_count == n


failed = [
    r.id for r in meta.itertuples() if not extract_frames(r.path, FRAMES30_DIR / r.id)
]
print("Failed:", len(failed), failed[:10])


Failed: 0 []


In [6]:
prep = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

def make_extractor():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier = nn.Identity()  # Output: 1280-dimensional feature vector
    return m.eval().to(device)

def frame_paths(vid):
    ps = sorted((FRAMES30_DIR / vid).glob("*.jpg"))
    assert len(ps) == N_FRAMES, f"{vid}: {len(ps)} frames"
    return ps

def pixel_motion(paths):
    g = [
        np.asarray(Image.open(p).convert("L").resize((64, 64)), dtype=np.float32) / 255
        for p in paths
    ]
    g = np.stack(g)
    return np.abs(np.diff(g, axis=0)).mean(axis=(1, 2))  # [29]

@torch.no_grad()
def build_cache():
    fc = CACHE_DIR / "effb0_30f_feats.npy"
    pc = CACHE_DIR / "effb0_30f_pix.npy"
    if fc.exists() and pc.exists():
        F, P = np.load(fc), np.load(pc)
        if len(F) == len(meta):
            return F, P
    model = make_extractor()
    Fs, Ps = [], []
    for i, vid in enumerate(meta["id"]):
        ps = frame_paths(vid)
        x = torch.stack([prep(Image.open(p).convert("RGB")) for p in ps]).to(device)
        Fs.append(model(x).cpu().numpy())
        Ps.append(pixel_motion(ps))
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(meta)}")
    F = np.stack(Fs).astype(np.float32)  # [N, 30, 1280]
    P = np.stack(Ps).astype(np.float32)  # [N, 29]
    np.save(fc, F)
    np.save(pc, P)
    return F, P

F_all, P_all = build_cache()
print(F_all.shape, P_all.shape)


(150, 30, 1280) (150, 29)


In [7]:
# ---- Grouping from solution.csv (before anything else) ----
ldf = pd.read_csv(LABELS_CSV, dtype={"id": str})
group_map = dict(zip(ldf["id"].str.zfill(5), ldf["group"]))
meta["group"] = meta["id"].map(group_map)
print(
    "Empty groups:",
    meta["group"].isna().sum(),
    "| Number of groups:",
    meta["group"].nunique(),
)

TOP_K = 8
C_GRID = np.logspace(-5, 2, 8)


def build_inputs(F, P, k=TOP_K):
    # F: [N, 30, 1280], P: [N, 29]
    mean_feat = F.mean(axis=1)  # [N, 1280]
    # Motion for each frame pair is attributed to the second frame, so +1
    top_idx = np.argsort(-P, axis=1)[:, :k] + 1  # [N, k]
    topk_feat = np.stack([F[i, top_idx[i]].mean(axis=0) for i in range(len(F))])
    motion = np.stack([P.mean(1), P.max(1), np.sort(P, 1)[:, -k:].mean(1)], axis=1)
    return np.hstack([mean_feat, topk_feat, motion]).astype(np.float32)


X_all = build_inputs(F_all, P_all)
y_all = meta["label"].values
groups_all = meta["group"].values
split_all = meta["split"].values

tr_mask = split_all == "train"
va_mask = split_all == "val"
te_mask = split_all == "test"


def make_clf():
    return make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=C_GRID,
            cv=3,
            scoring="average_precision",
            class_weight="balanced",
            max_iter=5000,
            n_jobs=-1,
        ),
    )


def mean_group_ap(y, score, grp):
    # Mean AP per group; single-class groups are ignored
    aps = []
    for g in np.unique(grp):
        m = grp == g
        if len(np.unique(y[m])) == 2:
            aps.append(average_precision_score(y[m], score[m]))
    return float(np.mean(aps)) if aps else float("nan")


def fit_eval(train_idx, eval_idx):
    clf = make_clf().fit(X_all[train_idx], y_all[train_idx])
    s = clf.predict_proba(X_all[eval_idx])[:, 1]
    return {
        "AP": average_precision_score(y_all[eval_idx], s),
        "groupAP": mean_group_ap(y_all[eval_idx], s, groups_all[eval_idx]),
        "C": float(clf[-1].C_[0]),
    }


def run_cv(n_splits=5, n_repeats=3, seed=0):
    idx = np.where(tr_mask | va_mask)[0]
    res = []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + r)
        for tr, ev in skf.split(idx, y_all[idx]):
            res.append(fit_eval(idx[tr], idx[ev]))
    ap = np.array([r["AP"] for r in res])
    print(f"CV AP: {ap.mean():.3f} ± {ap.std():.3f}  (n={len(res)})")
    print("Selected C values:", sorted({r["C"] for r in res}))
    return res


# Simple run: train on train set, evaluate on validation set
simple = fit_eval(np.where(tr_mask)[0], np.where(va_mask)[0])
print("val:", simple)
res = run_cv()

Empty groups: 0 | Number of groups: 1
val: {'AP': 0.752499568101072, 'groupAP': 0.752499568101072, 'C': 0.001}
CV AP: 0.685 ± 0.040  (n=15)
Selected C values: [1e-05, 0.0001, 0.001, 0.01, 0.1, 100.0]


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

D = 1280
blocks = {
    "motion only (3)": slice(2 * D, 2 * D + 3),
    "mean only (1280)": slice(0, D),
    "topk only (1280)": slice(D, 2 * D),
    "all (2563)": slice(0, 2 * D + 3),
}


def cv_ap(cols, C=0.01, pca=None, n_splits=5, n_repeats=3, seed=0):
    idx = np.where(tr_mask | va_mask)[0]
    steps = [StandardScaler()]
    if pca:
        steps.append(PCA(n_components=pca, random_state=0))
    steps.append(LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
    aps = []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + r)
        for tr, ev in skf.split(idx, y_all[idx]):
            clf = make_pipeline(*steps).fit(X_all[idx[tr]][:, cols], y_all[idx[tr]])
            s = clf.predict_proba(X_all[idx[ev]][:, cols])[:, 1]
            aps.append(average_precision_score(y_all[idx[ev]], s))
    return np.mean(aps), np.std(aps)


for name, cols in blocks.items():
    m, s = cv_ap(cols)
    print(f"{name:20s} AP = {m:.3f} ± {s:.3f}")

m, s = cv_ap(blocks["all (2563)"], pca=32)
print(f"{'all + PCA32':20s} AP = {m:.3f} ± {s:.3f}")

motion only (3)      AP = 0.619 ± 0.039
mean only (1280)     AP = 0.740 ± 0.065
topk only (1280)     AP = 0.663 ± 0.043
all (2563)           AP = 0.701 ± 0.050
all + PCA32          AP = 0.683 ± 0.047


In [9]:
def cv_ap_grid(cols, n_splits=5, n_repeats=3, seed=0):
    idx = np.where(tr_mask | va_mask)[0]
    aps, chosen = [], []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + r)
        for tr, ev in skf.split(idx, y_all[idx]):
            clf = make_pipeline(
                StandardScaler(),
                LogisticRegressionCV(
                    Cs=C_GRID,
                    cv=3,
                    scoring="average_precision",
                    class_weight="balanced",
                    max_iter=5000,
                    n_jobs=-1,
                ),
            ).fit(X_all[idx[tr]][:, cols], y_all[idx[tr]])
            s = clf.predict_proba(X_all[idx[ev]][:, cols])[:, 1]
            aps.append(average_precision_score(y_all[idx[ev]], s))
            chosen.append(float(clf[-1].C_[0]))
    print("Selected C values:", sorted(set(chosen)))
    return np.mean(aps), np.std(aps)


m, s = cv_ap_grid(blocks["mean only (1280)"])
print(f"{'mean only, C tuned':20s} AP = {m:.3f} ± {s:.3f}")

Selected C values: [1e-05, 0.0001, 0.001, 0.01, 0.1, 1.0, 100.0]
mean only, C tuned   AP = 0.719 ± 0.059
